# Prompting Techniques with Ollama

Prompting steers a pretrained LLM at inference time, with **no weight updates**. In this notebook you will:

1. See four techniques in action: **few-shot**, **chain-of-thought**, **self-consistency**, **RAG**
2. Practice them yourself in short exercises

| Technique | Idea |
|---|---|
| Zero/Few-shot | Give instructions, optionally with examples |
| Chain-of-Thought (CoT) | Ask for step-by-step reasoning before the answer |
| Self-Consistency | Sample several CoT chains, majority-vote the answer |
| RAG | Retrieve relevant documents and put them in the prompt |

## Setup
This notebook runs locally with the project's Python 3.11 environment and your installed Ollama app. Start Ollama before running the notebook. The default model is `gemma4:e2b`; override it by setting the `OLLAMA_MODEL` environment variable before starting Jupyter.

From the repository root on Windows PowerShell:

```powershell
.\.venv\Scripts\Activate.ps1
python -m jupyter lab
```

In [1]:
import os, re, sys
from collections import Counter

import ollama

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        f"This project expects Python 3.11; this kernel uses {sys.version.split()[0]}. "
        "Select .venv\\Scripts\\python.exe as the notebook kernel."
    )

MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
try:
    installed_models = [item.model for item in ollama.list().models]
except ConnectionError as exc:
    raise RuntimeError("Cannot connect to Ollama. Start the Ollama app, then rerun this cell.") from exc

if MODEL not in installed_models:
    raise RuntimeError(
        f"Ollama model '{MODEL}' is not installed. Run: ollama pull {MODEL}"
    )

print(f"Python {sys.version.split()[0]} | Ollama connected | model: {MODEL}")

Python 3.11.9 | Ollama connected | model: gemma4:e2b


In [2]:
def generate(prompt, max_tokens=150, temperature=0.0):
    """Send a prompt to the model and return its text. temperature=0 is deterministic."""
    r = ollama.generate(model=MODEL, prompt=prompt, think=False,
                        options={"temperature": temperature, "num_predict": max_tokens})
    return r["response"].strip()

print(generate("Say hello in one short sentence."))

Hello!


## Demo 1: Zero-shot vs. Few-shot
Few-shot adds input→output examples so the model copies the format.

In [3]:
review = "The pacing was dreadful, but the soundtrack was gorgeous."
task = "Classify the review as Positive, Negative, or Mixed. Reply with one word."

zero_shot = f"{task}\n\nReview: {review}\nLabel:"

few_shot = f"""{task}

Review: A masterpiece of modern cinema.
Label: Positive

Review: I fell asleep halfway through.
Label: Negative

Review: Great acting, but the plot made no sense.
Label: Mixed

Review: {review}
Label:"""

print("Zero-shot:", generate(zero_shot, max_tokens=10))
print("Few-shot :", generate(few_shot, max_tokens=10))

Zero-shot: Mixed
Few-shot : Mixed


## Demo 2: Chain-of-Thought
Asking the model to reason step by step helps on multi-step problems.

In [4]:
question = ("A store had 50 shirts. It sold 18 on Monday, got a shipment of 30, "
            "then sold half of what it had. How many shirts are left?")   # answer: 31

print("Direct:\n", generate(f"Q: {question}\nGive only the final number.\nA:", max_tokens=10))
print("\nZero-shot CoT:\n", generate(f"Q: {question}\nA: Let's think step by step.", max_tokens=200))

Direct:
 36

Zero-shot CoT:
 Here is the step-by-step solution:

**1. Starting Inventory:**
The store begins with **50** shirts.

**2. Sales on Monday:**
They sold 18 shirts.
$50 - 18 = 32$ shirts remaining.

**3. Shipment Received:**
They received a shipment of 30 shirts.
$32 + 30 = 62$ shirts total.

**4. Second Sale:**
They sold half of what they had (half of 62).
Half of 62 is $62 / 2 = 31$ shirts sold.

**5. Final Count:**
Subtract the second sale from the current total:
$62 - 31 = 31$ shirts left.

**Answer:** There are **31** shirts left.


## Demo 3: Self-Consistency
Sample several reasoning chains (temperature > 0) and take the most common final answer.

In [5]:
def final_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None

prompt = (f"Q: {question}\nThink step by step, then end with 'The answer is <number>.'\nA:")
answers = []
for _ in range(5):
    out = generate(prompt, max_tokens=250, temperature=0.8)
    answers.append(final_number(out))

print("Votes :", Counter(answers))
print("Winner:", Counter(answers).most_common(1)[0][0], "(truth: 31)")

Votes : Counter({'31': 5})
Winner: 31 (truth: 31)


## Demo 4: RAG
Retrieve the most relevant document (TF-IDF here; real systems use embeddings + a vector DB) and add it to the prompt.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "Acme Corp's refund policy: items can be returned within 45 days with a receipt.",
    "Acme Corp support hours are 9am-5pm ICT, Monday to Friday.",
    "Acme Corp's Pro plan costs $29 per month and includes 5 seats.",
    "Acme Corp headquarters are located in Phnom Penh, Cambodia.",
]
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

def retrieve(query, k=1):
    scores = cosine_similarity(vec.transform([query]), doc_vecs).ravel()
    return [docs[i] for i in scores.argsort()[::-1][:k]]

def rag_answer(query, k=1):
    context = "\n".join(retrieve(query, k))
    return generate(f"Answer using only the context.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:")

q = "How many days do I have to return an item at Acme Corp?"
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q))

No context: I do not have access to specific return policies for Acme Corp. To find out how many days you have to return an item, you will need to check their official website, return policy page, or contact Acme Corp directly.
With RAG  : The context does not state how many days you have to return an item at Acme Corp.


---
# Hands-on Practice
The exercises below include worked solutions. Re-run each cell and compare outputs. Small models are noisy, so try a few variations.

### Exercise 1: Few-shot extraction
Make the model turn a sentence into JSON with keys `name`, `city`, `age`. First try zero-shot, then add **2 examples** to get a clean, consistent format.

In [7]:
text = "Sokha, a 29-year-old engineer, just moved to Siem Reap."

zero_shot_prompt = f"""Extract name, city, and age as JSON.
Return only valid JSON.

Text: {text}
JSON:"""

few_shot_prompt = f"""Extract name, city, and age as JSON.
Return only valid JSON with keys in this order: name, city, age.

Text: Dara is 34 years old and lives in Phnom Penh.
JSON: {{"name": "Dara", "city": "Phnom Penh", "age": 34}}

Text: Lina, age 22, moved to Battambang.
JSON: {{"name": "Lina", "city": "Battambang", "age": 22}}

Text: {text}
JSON:"""

print("Zero-shot:", generate(zero_shot_prompt, max_tokens=60))
print("Few-shot :", generate(few_shot_prompt, max_tokens=60))

Zero-shot: ```json
{
  "name": "Sokha",
  "city": "Siem Reap",
  "age": 29
}
```
Few-shot : {"name": "Sokha", "city": "Siem Reap", "age": 29}


### Exercise 2: Chain-of-Thought
The puzzle below often trips up direct answers. Compare a **direct** prompt with a **CoT** prompt. Which is right? (Answer: 0.05, i.e. 5 cents)

In [8]:
puzzle = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much is the ball?"

direct = f"Question: {puzzle}\nReply with only the dollar amount.\nAnswer:"
cot = (f"Question: {puzzle}\nSolve it algebraically step by step. "
       "Check that the two prices sum to $1.10, then end with 'Answer: <amount>'.")

print("Direct:", generate(direct, max_tokens=40))
print("CoT   :", generate(cot, max_tokens=160))

Direct: 0.05
CoT   : Here is the step-by-step algebraic solution:

**1. Define the variables:**
Let $B$ be the cost of the bat.
Let $L$ be the cost of the ball.

**2. Set up the equations based on the problem statement:**
We are given two pieces of information, which we can translate into two equations:

*   **Equation 1 (Total cost):** The bat and the ball cost $1.10 in total.
    $$B + L = 1.10$$

*   **Equation 2 (Cost difference):** The bat costs $1.00 more than the ball.
    $$B = L + 1.00$$

**3. Solve the system of equations


### Exercise 3: Self-Consistency
Write a function `self_consistent(prompt, k)` that samples `k` answers at `temperature=0.8` and returns the majority answer. Test it on the puzzle above with `k=1` vs `k=7`. Does voting help?

*Hint:* ask the model to end with `Answer: <number>` so extracting the answer is easy.

In [9]:
def self_consistent(prompt, k=5):
    if k < 1:
        raise ValueError("k must be at least 1")

    answers = []
    for _ in range(k):
        output = generate(prompt, max_tokens=160, temperature=0.8)
        answer = final_number(output)
        if answer is not None:
            answers.append(answer)

    if not answers:
        raise ValueError("The model did not produce a numeric answer.")
    return Counter(answers).most_common(1)[0][0]

cot_prompt = (f"Question: {puzzle}\nReason step by step, verify the total, "
              "and end with 'Answer: <number>'.")
print("k=1:", self_consistent(cot_prompt, k=1))
print("k=7:", self_consistent(cot_prompt, k=7))

k=1: 2
k=7: 2


### Exercise 4: RAG
1. Add **two new facts** of your own to `docs` (re-fit the vectorizer after).
2. Ask a question that only your facts can answer.
3. Compare the answer with and without retrieval. What happens when the answer is **not** in the docs? Edit the RAG prompt so the model says *"I don't know"* in that case.

In [10]:
docs.extend([
    "Techpreneur Bootcamp's Khmer OCR workshop is held on Saturday in Phnom Penh.",
    "The Lao OCR demo recognizes printed Lao text and returns Unicode text.",
])
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

def rag_answer(query, k=1):
    context = "\n".join(retrieve(query, k))
    prompt = f"""Answer using only the context below.
If the context does not contain the answer, reply exactly: I don't know

Context:
{context}

Question: {query}
Answer:"""
    return generate(prompt, max_tokens=60)

q = "When is Techpreneur Bootcamp's Khmer OCR workshop?"
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q))

missing_q = "Who is Acme Corp's CEO?"
print("Missing fact:", rag_answer(missing_q))

No context: Please provide more context or information. I do not have access to a schedule or specific event details for "Techpreneur Bootcamp's Khmer OCR workshop."
With RAG  : Saturday
Missing fact: I don't know


### Challenge (optional)
Pick a task from your own work (e.g. classify support tickets, summarize, translate to Khmer). Design a prompt, then improve it by adding **a role, a format constraint, and examples**. Note which change helped most.

---
## Recap
- **Few-shot**: examples fix format and style
- **CoT**: reasoning steps help multi-step problems
- **Self-consistency**: voting across samples improves reliability, at k× the cost
- **RAG**: grounds answers in your own data

Next: when prompting is not enough, **fine-tuning** updates the weights.